In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark").getOrCreate()
data = [
(1,"Ravi","IT",25000),
(2,"Ramu","HR",30000),
(3,"Kumar","IT",35000),
(4,None,"Finance",40000)
]

columns = [
"id",
"name",
"department",
"salary"
]

df = spark.createDataFrame(
data,
columns
)

df.show()

+---+-----+----------+------+
| id| name|department|salary|
+---+-----+----------+------+
|  1| Ravi|        IT| 25000|
|  2| Ramu|        HR| 30000|
|  3|Kumar|        IT| 35000|
|  4| NULL|   Finance| 40000|
+---+-----+----------+------+



In [0]:
spark.sql("SELECT current_catalog()").show()

spark.sql("SHOW SCHEMAS").show()

spark.sql("SHOW VOLUMES").show()

+-----------------+
|current_catalog()|
+-----------------+
|        workspace|
+-----------------+

+------------------+
|      databaseName|
+------------------+
|           default|
|information_schema|
+------------------+

+--------+-----------+
|database|volume_name|
+--------+-----------+
+--------+-----------+



In [0]:
df.write \
.mode("overwrite") \
.saveAsTable("default.bronze_employees")

In [0]:
silver_df = df.na.drop()

In [0]:
silver_df.write \
.mode("overwrite") \
.saveAsTable("default.silver_employees")

In [0]:
spark.table(
    "default.silver_employees"
).show()

+---+-----+----------+------+
| id| name|department|salary|
+---+-----+----------+------+
|  1| Ravi|        IT| 25000|
|  2| Ramu|        HR| 30000|
|  3|Kumar|        IT| 35000|
+---+-----+----------+------+



In [0]:
from pyspark.sql.functions import sum

gold_df = silver_df.groupBy(
    "department"
).agg(
    sum("salary").alias("total_salary")
)

In [0]:
gold_df.write \
.mode("overwrite") \
.saveAsTable("default.gold_employees")

In [0]:
spark.table(
    "default.gold_employees"
).show()

+----------+------------+
|department|total_salary|
+----------+------------+
|        IT|       60000|
|        HR|       30000|
+----------+------------+



In [0]:
spark.sql("SHOW TABLES").show()

+--------+----------------+-----------+
|database|       tableName|isTemporary|
+--------+----------------+-----------+
| default|bronze_employees|      false|
| default|  gold_employees|      false|
| default|silver_employees|      false|
+--------+----------------+-----------+

